##### 9. Lag structure and autocorrelation checks

Calculate lag correlations for shortlisted NMIs at 1, 2, 48, and 336 half-hour lags.

In [1]:
import pandas as pd
from pathlib import Path

# Load half-hourly NMI consumption data.
df = pd.read_parquet(Path("../../Datasets/LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq"))
df = df.drop(columns=["6102507141 consumption", "VAAA003225 consumption"])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

In [2]:
# Reuse the shortlist generated by Task/MAST90106 Group 2 EDA.ipynb.
summary_df = pd.read_csv(Path("../EDA_SummaryByNMI.csv"), index_col=0)
NMI_shortlist = summary_df[((summary_df['Status']=='Active') | (summary_df['Status']=='Mostly Active'))].index
print(f'Number of shortlisted NMIs: {len(NMI_shortlist)}')
NMI_shortlist

Number of shortlisted NMIs: 95


Index(['6102000812', '6102002302', '6102005454', '6102005592', '6102009742',
       '6102009743', '6102009744', '6102023971', '6102038376', '6102046251',
       '6102047562', '6102079015', '6102126219', '6102136796', '6102159807',
       '6102241120', '6102253330', '6102284820', '6102329966', '6102332526',
       '6102479831', '6102548873', '6102573328', '6102798810', '6102823324',
       '6103002422', '6103004482', '6103005867', '6103005869', '6103009639',
       '6103010081', '6103010326', '6103011168', '6103015873', '6103022015',
       '6103022017', '6103022018', '6103029662', '6103029663', '6103031269',
       '6103031796', '6103054578', '6103054611', '6103055142', '6103055392',
       '6103055643', '6103056620', '6103056621', '6103056622', '6103056625',
       '6103063019', '6103063020', '6103065120', '6103065121', '6103065471',
       '6103066694', '6103067996', '6103068525', '6103077259', '6203397519',
       '6203397522', '6203848319', '6203949247', 'VAAA000057', 'VAAA000173',

In [3]:
# Add one exact lagged version of a variable.
def add_lag(data, variable, n):
    data[f"lag_{n}"] = data[variable].shift(n)

In [4]:
# Compute current-vs-past correlations for each shortlisted NMI.
lags = [1, 2, 48, 336]
rows = []

for nmi in NMI_shortlist:
    variable = f"{nmi} consumption"
    data = df[[variable]].copy()
    for n in lags:
        add_lag(data, variable, n)
    rows.append([nmi, *data.corr().loc[variable, [f"lag_{n}" for n in lags]]])

corr_table = pd.DataFrame(rows, columns=["NMI", *[f"lag_{n}" for n in lags]]).round(3)
corr_table

,NMI,lag_1,lag_2,lag_48,lag_336
0,6102000812,0.979,0.960,0.799,0.870
1,6102002302,0.978,0.942,0.691,0.877
2,6102005454,0.989,0.971,0.828,0.922
3,6102005592,0.967,0.925,0.744,0.724
4,6102009742,0.766,0.600,0.627,0.685
...,...,...,...,...,...
90,VAAA004066,0.909,0.829,0.610,0.572
91,VCCCAE0035,0.955,0.914,0.682,0.669
92,VCCCBC0096,0.966,0.934,0.873,0.852
93,VCCCSC0045,0.978,0.946,0.738,0.849


In [5]:
# Label the strongest lag horizon.
meaning = {
    "1": "half-hour",
    "2": "one-hour",
    "48": "day",
    "336": "week",
}

def interpret(row):
    lag = row[[f"lag_{n}" for n in lags]].abs().idxmax().removeprefix("lag_")
    return meaning[lag]

interpretation = corr_table.assign(interpretation=corr_table.apply(interpret, axis=1))[["NMI", "interpretation"]]
interpretation

,NMI,interpretation
0,6102000812,half-hour
1,6102002302,half-hour
2,6102005454,half-hour
3,6102005592,half-hour
4,6102009742,half-hour
...,...,...
90,VAAA004066,half-hour
91,VCCCAE0035,half-hour
92,VCCCBC0096,half-hour
93,VCCCSC0045,half-hour


In [6]:
# Count which lag is strongest across the shortlist.
corr_table[[f"lag_{n}" for n in lags]].abs().idxmax(axis=1).value_counts()

lag_1      93
lag_336     2
Name: count, dtype: int64